<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Train Test Split and Cross Validation - Solution

---

### About
An introduction to creating generalizable models using methods such as train, test, split, and describing the bias variance tradeoff.


### Learning Objectives
- Describe the principles of Generalizability.
- Compare Bias and Variance and describe the tradeoff between when selecting a model.
- Define and apply the Train Test Split method using `scikit-learn`.
- Explore and apply the k-Folds Cross Validation method in `scikit-learn`.


### Notebook Guide

- Creating Models That Are Generalizable
- Part I: The Train-Test Split
- Now You Try
- Part II: Cross Validation
- Now You Try
- Conclusion and Takeaways

In [10]:
# import libraries we will use

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.linear_model import LinearRegression

from sklearn.model_selection import train_test_split, cross_val_score, KFold

from sklearn.metrics import mean_squared_error

# Creating Models That Are Generalizable
---

## Overfitting 

We run into a problem when powerful models can perfectly fit the data on which they are trained. These models are **low bias** and **high variance**. Because we only know how the model fits the data we have, rather than the underlying population of interest, we don't have a good picture of how it work with new (aka: "outside") data or "in the real world".
<br /><br />
![overfit](../assets/overfit.png)

**Solution:** Create a procedure that estimates how well a model is likely to perform on out-of-sample data and use that to choose between models. Overall, our model will likely have poorer performance on our data sample, but will be more generalizable or work better overall. 


## Making a Prediction From a Sample

We often assume that our data set contains every sample for a given problem. In other words, the whole population.  In cases like the Titanic data set, that is true. Although it's convenient, in the real world all you typically have is a **sample** of data. This may sound like a contentious statement, but the point of machine learning is to generalize from a sample of data to the underlying population. If you already have data for an entire population, then you have no need for machine learning: you can apply statistics directly and get optimal answers!

So how can we get our models to generalize well?

**Methods:**   
1. **Train-test-split**:
     - Is useful if cross-validation is not practical (e.g. it takes too long to train)
     - It is also useful for computing a quick confusion matrix (for classification models)
     - You could also use this as a final step after the model is finalized (often called evaluating the model against a validation set)
1. **Cross-validation** is the gold standard for estimating accuracy and comparing accuracy across models for many model types.
1. **Three-way split** combines cross-validation and the train-test-split. It takes an initial split to be used as a final validation set, then uses cross-validation on the rest.

# Part I: The Train-Test Split
---

![](../assets/train_test.png)

1. Split the data set into two pieces: a **training set** and a **testing set**.
2. Train the model on the **training set**.
3. Test the model on the **testing set** and evaluate how well we did.

Often a good rule-of-thumb is 70% training / 30% test, but this can vary based on the size of your data set. For example, with a small data set you would need to use as much training data as possible (in return, your test accuracy will be more variable).

What does this accomplish?

- Models can be trained and tested on **different data** (we treat testing data like out-of-sample data).
- Response values are known for the testing set and thus **predictions can be evaluated**.

In [11]:
# read in the bikeshare data

data = pd.read_csv("../data/bikeshare.csv")
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [12]:
# split the data

train, test = train_test_split(data)

In [13]:
# inspect the shape of the data that has been split

print("Full data set shape:", data.shape)
print("Train data set shape:", train.shape, "% of Total:", round(len(train)/len(data)*100, 2))
print("Test data set shape:", test.shape, "% of Total:", round(len(test)/len(data)*100, 2))
# this gives you a 75:25 split

Full data set shape: (17379, 17)
Train data set shape: (13034, 17) % of Total: 75.0
Test data set shape: (4345, 17) % of Total: 25.0


In [14]:
data.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='object')

In [15]:
# define a feature matrix for a model

X = data[['weathersit', 'temp', 'hum', 'windspeed']]
y = data['casual']

In [16]:
# we often read in features so that we can do the split and have X and y features set up for us

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [17]:
# inspect the outputs

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (13034, 4)
y_train shape: (13034,)
X_test shape: (4345, 4)
y_test shape: (4345,)


In [10]:
# sometimes you need to (approximately) keep the proportion of y values through the training and test sets
# to do this, use the stratify parameter

#  Now You Try
Use the `scikit-learn` [docs on **Train Test Split**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to help you:
1. **create a train test split with a 70:30 split on X and y**.
2. **return the same split everytime you run the function (use the value `1653` for this parameter)**.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=1653)

3.**Verify the splitted data dimention matches the original dataframe**

In [19]:
print("Full shape:", data.shape)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

Full shape: (17379, 17)
X_train shape: (12165, 4)
y_train shape: (12165,)
X_test shape: (5214, 4)
y_test shape: (5214,)


### Fit a Linear Regression model on the training data

In [24]:
# instanitate your Linear Regression  
model = LinearRegression()

In [25]:
# fit your model on the features (X) and the target (y)
model.fit(X_train, y_train) #set to train

LinearRegression()

### Check the metrics on the training and the test data

In [26]:
# Regression Metrics - training

y_pred = model.predict(X_train)
print("MSE", round(mean_squared_error(y_train, y_pred), 3))
print("R Sq", round(model.score(X_train, y_train), 3))

MSE 1750.083
R Sq 0.307


In [27]:
# Regression Metrics - testing

y_pred = model.predict(X_test)
print("MSE", round(mean_squared_error(y_test, y_pred), 3))
print("R Sq", round(model.score(X_test, y_test), 3))

MSE 1494.968
R Sq 0.323


The model performs better on the test data than the training data (this can happen).

---

To the slides!

---

# Part II: Cross Validation
---

Train/test split provides us with helpful tool, but it's a shame that we are tossing out a large chunk of our data for testing purposes.

**How can we use the maximum amount of our data points while still ensuring model integrity?**

1. Split our data into a number of different pieces (folds).
2. Train using `k-1` folds for training and a different fold for testing.
3. Average our model against EACH of those iterations.
4. Choose our model and TEST it against the final fold.
5. Average all test accuracies to get the estimated out-of-sample accuracy.

Although this may sound complicated, we are just training the model on k separate train-test-splits, then taking the average of the resulting test accuracies!

![](../assets/cross_validation_diagram.png)

### Cross-validation score 
`cross_val_score` makes this a lot easier to code: [docs](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)

In [29]:
# Create a 5-fold cross-validation splitter
kf = KFold(
    n_splits=5,        # Split the training data into 5 folds
    shuffle=True,      # Shuffle the rows once before creating the folds
    random_state=1318  # Make the shuffle reproducible
)

# Create a Linear Regression model
lm = LinearRegression()

In [30]:
# Run cross_val_score and analyse the variable score

scores = cross_val_score(lm, X, y, cv=kf)
scores

array([0.33368212, 0.31997683, 0.31450627, 0.29330711, 0.29896289])

In [31]:
# Print the average R square score
print(f'CV R Square {scores}, \nAverage R Square {scores.mean()}')

CV R Square [0.33368212 0.31997683 0.31450627 0.29330711 0.29896289], 
Average R Square 0.31208704409085397


Don't want to to use **R-square** for your score?  Using this for a different model type? Check out your scoring options [here](https://scikit-learn.org/stable/modules/model_evaluation.html)

In [33]:
# Run cross_val_score using RMSE as the metric

rmse_scores = cross_val_score(lm, X, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_scores

array([-38.64136923, -39.05844574, -41.67657568, -43.68951745,
       -41.2925134 ])

In [34]:
# Print the average R square score
print(f'CV RMSE {rmse_scores}, \nAverage R RMSE {rmse_scores.mean()}')

CV RMSE [-38.64136923 -39.05844574 -41.67657568 -43.68951745 -41.2925134 ], 
Average R RMSE -40.87168429935912


#  Now You Try 
1. Run a Linear Regression using cross validation with `cross_val_score` with k folds:
    - Creating 10 splits
    - Evaluating on Mean Absolute error
    - Using the same random state value as before
   
2. Do you reach the same conclusion as running a cross validation with 5 folds?

#### Run a Linear Regression  using cross validation with k folds creating 10 splits

In [39]:
kf = KFold(n_splits=10, shuffle=True, random_state= 1318)

scores = cross_val_score(lm, X, y, cv=kf, scoring = "neg_mean_absolute_error")

print(f'CV MAE {scores}, \nAverage MAE {scores.mean()}')

CV MAE [-25.19655944 -24.64904897 -25.01362223 -25.16065708 -27.00775542
 -25.79143085 -27.57025938 -27.84213506 -25.95793321 -25.37533632], 
Average MAE -25.956473796843568


#### Run a Linear Regression  using cross validation with k folds creating 5 splits

In [40]:
kf = KFold(n_splits=5, shuffle=True, random_state= 1318)

scores = cross_val_score(lm, X, y, cv=kf, scoring = "neg_mean_absolute_error")

print(f'CV MAE {scores}, \nAverage MAE {scores.mean()}')

CV MAE [-24.95097976 -25.10994046 -26.38690748 -27.67669117 -25.65774498], 
Average MAE -25.956452770410266


# Recap

1. Define overfitting, and underfitting for models
1. Describe 2 methods to make your models more generalizable
1. Apply train, test, split and K-Folds cross validations to bikeshare data